# AETHER BFCL official execution

Runs the pinned native official package on development-scoped preparation only.
This notebook does not access AETHER validation/test splits. It exports raw
official outputs plus a hash-verified AETHER wrapper bundle.


In [ ]:
from pathlib import Path
import os,shutil,subprocess,sys,tarfile,hashlib
WORK=Path('/kaggle/working'); RUNTIME=Path('/tmp/aether-bfcl')
REPO=RUNTIME/'aether'; VENV=RUNTIME/'venv'; PY=VENV/'bin'/'python'
OUTPUT=WORK/'aether-bfcl-official'; COMMIT='7ffad6349aae998ed94cfaa526fbfe103549926f'
URL='https://github.com/ama1a005/aether.git'
os.environ.update({'PYTHONNOUSERSITE':'1','MPLBACKEND':'Agg'})
RUNTIME.mkdir(parents=True,exist_ok=True)


In [ ]:
if shutil.which('uv') is None: subprocess.run([sys.executable,'-m','pip','install','uv'],check=True)
UV=shutil.which('uv'); env=os.environ.copy(); env['UV_PYTHON_INSTALL_DIR']=str(RUNTIME/'uv-python')
subprocess.run([UV,'venv','--python','3.11.9',str(VENV)],check=True,env=env)
subprocess.run(['git','clone','--branch','phase1/core-nov19','--single-branch',URL,str(REPO)],check=True)
subprocess.run(['git','fetch','origin',COMMIT],cwd=REPO,check=True)
subprocess.run(['git','checkout','--detach',COMMIT],cwd=REPO,check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()==COMMIT
assert not subprocess.check_output(['git','status','--porcelain'],cwd=REPO,text=True).strip()


In [ ]:
subprocess.run([UV,'pip','install','--python',str(PY),'-r',str(REPO/'requirements-lock.txt')],check=True,env=env)
subprocess.run([UV,'pip','install','--python',str(PY),'bfcl_eval==2026.3.23'],check=True,env=env)
subprocess.run([UV,'pip','install','--python',str(PY),'--no-deps','-e',str(REPO)],check=True,env=env)
subprocess.run([str(PY),'-m','pytest','tests/unit/test_external_bundle.py','tests/unit/test_native_benchmark_runners.py','tests/unit/test_native_official_package.py','-q'],cwd=REPO,check=True)


In [ ]:
command=[str(PY),'-m','benchmarks.run_native_official','bfcl','--output',str(OUTPUT)]
command += ['--snapshot-manifest','/kaggle/working/model_snapshot_manifest.json']
subprocess.run(command,cwd=REPO,check=True)
subprocess.run([str(PY),'-m','scripts.verify_external_bundle',str(OUTPUT),'--expected-kind','bfcl_official'],cwd=REPO,check=True)
archive=WORK/'aether-bfcl-official-results.tar.gz'
with tarfile.open(archive,'w:gz') as bundle: bundle.add(OUTPUT,arcname=OUTPUT.name)
print(archive,hashlib.sha256(archive.read_bytes()).hexdigest())
